# ML-08 — Train the Model
**Lane 2 — Refresh / Content Opportunity Scoring**

Trains Logistic Regression and Random Forest on the same features and evaluates them
on a **client-holdout** split, matching the metric (Precision@50, Precision@20, ROC AUC)
established for the baseline in `w04_baseline_score.ipynb`. Target: `is_declining_label`
(1 if `trend_direction == "down"`).

In [1]:
import pandas as pd, numpy as np, json
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
for c in ["impressions_90d","clicks_90d","sessions_90d","ai_sessions_90d"]:
    df[f"log_{c}"] = np.log1p(df[c].clip(lower=0))
print(df.shape, "| base rate:", df["is_declining_label"].mean().round(3))

(30000, 49) | base rate: 0.542


## Feature list

Deliberately excludes the label itself, the raw 30-day windows the label is computed
from, and both ID columns (used only for grouping).

In [2]:
NUMERIC_FEATURES = [
    "search_volume","competition","cpc","word_count","char_count",
    "log_impressions_90d","log_clicks_90d","log_sessions_90d","log_ai_sessions_90d",
    "days_with_impressions","days_with_sessions","content_age_days",
    "days_since_last_update","ctr","avg_position","engagement_rate",
    "scroll_rate","ai_traffic_pct",
]
CATEGORICAL_FEATURES = [
    "competition_level","content_type","main_intent","age_tier",
    "freshness_tier","word_count_tier","impression_tier","position_tier",
]
FORBIDDEN = {"trend_direction","trend_pct","is_declining_label","is_declining",
             "impressions_last_30d","impressions_prev_30d","clicks_last_30d","clicks_prev_30d",
             "sessions_last_30d","sessions_prev_30d"}
assert FORBIDDEN.isdisjoint(set(NUMERIC_FEATURES) | set(CATEGORICAL_FEATURES)), "leakage in feature list!"
X_cols = NUMERIC_FEATURES + CATEGORICAL_FEATURES
print(f"{len(X_cols)} features, leakage check passed.")

26 features, leakage check passed.


## Client-holdout split

Grouped by `client_id`, not a plain random split — see `w06_validation_audit.ipynb`
for why the random-split alternative was rejected.

In [3]:
model_df = df.copy()
for c in NUMERIC_FEATURES:
    model_df[c] = pd.to_numeric(model_df[c], errors="coerce").fillna(0)
for c in CATEGORICAL_FEATURES:
    model_df[c] = model_df[c].fillna("unknown").astype(str)
y = model_df["is_declining_label"]
groups = model_df["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(model_df, y, groups=groups))
X_train, X_test = model_df.iloc[train_idx].copy(), model_df.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train: {len(X_train):,} rows / {X_train['client_id'].nunique()} clients")
print(f"Test:  {len(X_test):,} rows / {X_test['client_id'].nunique()} clients")
print(f"Base rate  train={y_train.mean():.3f}  test={y_test.mean():.3f}")

Train: 22,885 rows / 24 clients
Test:  7,115 rows / 8 clients
Base rate  train=0.550  test=0.517


In [4]:
pre = ColumnTransformer([
    ("num", StandardScaler(), NUMERIC_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
])

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

results = {"n_total": len(model_df), "n_train": len(X_train), "n_test": len(X_test),
           "n_train_clients": int(X_train["client_id"].nunique()),
           "n_test_clients": int(X_test["client_id"].nunique()),
           "train_base_rate": float(y_train.mean()), "test_base_rate": float(y_test.mean()),
           "models": {}}
fitted = {}
for name, clf in [
    ("logistic_regression", LogisticRegression(max_iter=10000, random_state=RANDOM_SEED)),
    ("random_forest", RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20,
                                              random_state=RANDOM_SEED, n_jobs=-1)),
]:
    pipe = Pipeline([("pre", pre), ("clf", clf)])
    pipe.fit(X_train[X_cols], y_train)
    proba = pipe.predict_proba(X_test[X_cols])[:, 1]
    results["models"][name] = {
        "roc_auc": roc_auc_score(y_test, proba),
        "precision_at_20": precision_at_k(y_test, proba, 20),
        "precision_at_50": precision_at_k(y_test, proba, 50),
    }
    fitted[name] = pipe
    X_test[f"proba_{name}"] = proba

for name, m in results["models"].items():
    print(f"{name:22} ROC AUC={m['roc_auc']:.3f}  P@20={m['precision_at_20']:.2f}  P@50={m['precision_at_50']:.2f}")

logistic_regression    ROC AUC=0.611  P@20=0.80  P@50=0.74
random_forest          ROC AUC=0.603  P@20=0.50  P@50=0.56


## Baseline, evaluated on the same held-out test set (apples-to-apples)

In [5]:
stale = (X_test["freshness_tier"] == "91-180").astype(int)
strikable = X_test["position_tier"].isin(["page_1","striking"]).astype(int)
visible = X_test["impression_tier"].isin(["moderate","good","excellent"]).astype(int)
baseline_score = X_test["impressions_90d"] * stale * strikable * visible

results["baseline_rule"] = {
    "roc_auc": roc_auc_score(y_test, baseline_score),
    "precision_at_20": precision_at_k(y_test, baseline_score.values, 20),
    "precision_at_50": precision_at_k(y_test, baseline_score.values, 50),
    "pct_flagged": float((baseline_score > 0).mean()),
}
print(results["baseline_rule"])

{'roc_auc': 0.5026313874386964, 'precision_at_20': 0.2, 'precision_at_50': 0.34, 'pct_flagged': 0.052986647926914966}


## Model vs. baseline — summary table

| Method | ROC AUC | P@20 | P@50 |
|---|---|---|---|
| Baseline rule | printed above | | |
| Random Forest | printed above | | |
| Logistic Regression | printed above | | |

Logistic Regression wins on every metric. See `w06_validation_audit.ipynb` for the
split-honesty check that explains why Random Forest's number looks worse than a naive
random-split evaluation would suggest, and `w07_action_playbook.ipynb` for the ranked
output built from the winning model.

## Feature importance & error analysis (winning model: Logistic Regression)

What the model actually found, and where it's wrong on the top-ranked pages —
used directly in the paper's Interpretation and Error Analysis sections.

In [6]:
feat_names = fitted['logistic_regression'].named_steps['pre'].get_feature_names_out()
coefs = fitted['logistic_regression'].named_steps['clf'].coef_[0]
top_lr_coefs = sorted(zip(feat_names, coefs), key=lambda x: -abs(x[1]))[:8]
print('Top Logistic Regression coefficients:')
for n, c in top_lr_coefs:
    print(f'  {n:40} {c:+.3f}')

importances = fitted['random_forest'].named_steps['clf'].feature_importances_
top_rf_importances = sorted(zip(feat_names, importances), key=lambda x: -x[1])[:8]
print('\nTop Random Forest importances:')
for n, v in top_rf_importances:
    print(f'  {n:40} {v:.4f}')

top50 = X_test.sort_values('proba_logistic_regression', ascending=False).head(50)
top50_tp = int(top50['is_declining_label'].sum())
top50_client_counts = top50['client_id'].value_counts().to_dict()
print(f"\nLogReg top-50: {top50_tp}/50 true positives")
print('Client concentration in top-50:', top50_client_counts)

error_analysis = {
    'top_lr_coefs': [[n, float(c)] for n, c in top_lr_coefs],
    'top_rf_importances': [[n, float(v)] for n, v in top_rf_importances],
    'top50_true_positives': top50_tp,
    'top50_client_counts': top50_client_counts,
}
json.dump(error_analysis, open('../artifacts/error_analysis.json', 'w'), indent=2)
print('\nsaved error_analysis.json')

Top Logistic Regression coefficients:
  num__log_impressions_90d                 +1.505
  cat__impression_tier_low                 +0.824
  cat__position_tier_top_3                 -0.818
  cat__impression_tier_excellent           -0.656
  cat__word_count_tier_1000-2000           +0.629
  num__log_clicks_90d                      -0.572
  cat__main_intent_unknown                 +0.538
  num__word_count                          +0.508

Top Random Forest importances:
  num__days_with_impressions               0.1751
  num__log_impressions_90d                 0.1263
  num__avg_position                        0.1009
  num__content_age_days                    0.0927
  num__word_count                          0.0525
  num__char_count                          0.0429
  cat__position_tier_top_3                 0.0331
  num__ctr                                 0.0291

LogReg top-50: 37/50 true positives
Client concentration in top-50: {'client_f369cb89fc': 22, 'client_d029fa3a95': 13, 'client_4e

In [7]:
Path("../artifacts").mkdir(parents=True, exist_ok=True)
json.dump(results, open("../artifacts/model_results.json", "w"), indent=2)
X_train.to_pickle("../artifacts/train_set.pkl")
X_test.to_pickle("../artifacts/test_set.pkl")
import pickle
pickle.dump(fitted, open("../artifacts/fitted_models.pkl", "wb"))
print("Saved model_results.json, train/test sets, and fitted models to work/artifacts/")

Saved model_results.json, train/test sets, and fitted models to work/artifacts/
